# 06. Binary Search Tree (BST)

A BST keeps this rule at **every node**: all values in the **left** subtree are **smaller**, all in the **right** subtree are **bigger**. So searching is like binary search, and **inorder traversal is sorted**.

**Learning order:** RULE → SEARCH → INSERT → DELETE → VALIDATE → KTH SMALLEST → LCA → SUCCESSOR → RANGE SUM → ITERATOR → INTERVIEW THINKING

## 1. Why BST?

Real life: a **phone contacts book** in a tree: from any name you know whether to go left (earlier in the alphabet) or right. Search, insert, delete cost **O(h)**: O(log n) if balanced, **O(n)** if the tree is skewed (inserting sorted data into a plain BST). Self-balancing trees (AVL, Red-Black) keep `h = O(log n)`. Java's `TreeMap` is one. Python has no built-in, so people use `bisect` or the `sortedcontainers` library.

In [1]:
from collections import deque

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def tree_from_list(vals):             # level order, None = missing child
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    q, i = deque([root]), 1
    while q and i < len(vals):
        node = q.popleft()
        if i < len(vals) and vals[i] is not None:
            node.left = TreeNode(vals[i]); q.append(node.left)
        i += 1
        if i < len(vals) and vals[i] is not None:
            node.right = TreeNode(vals[i]); q.append(node.right)
        i += 1
    return root

def level_order(root):
    if not root:
        return []
    q, res = deque([root]), []
    while q:
        res.append([])
        for _ in range(len(q)):
            node = q.popleft()
            res[-1].append(node.val)
            if node.left: q.append(node.left)
            if node.right: q.append(node.right)
    return res

def inorder(root):
    return inorder(root.left) + [root.val] + inorder(root.right) if root else []

def search(root, val):
    while root:
        if val == root.val:
            return root
        root = root.left if val < root.val else root.right
    return None

def insert(root, val):
    if not root:
        return TreeNode(val)
    if val < root.val:
        root.left = insert(root.left, val)
    else:
        root.right = insert(root.right, val)
    return root

t = None
for v in [5, 3, 7, 2, 4, 6, 8]:
    t = insert(t, v)
print(inorder(t), search(t, 4) is not None, search(t, 9))   # [2, 3, 4, 5, 6, 7, 8] True None

[2, 3, 4, 5, 6, 7, 8] True None


## 2. Delete (3 cases)

1. **Leaf:** just remove it.
2. **One child:** replace the node by its child.
3. **Two children:** replace the node's value with its **inorder successor** (smallest in the right subtree), then delete that successor from the right subtree.

In [2]:
def delete(root, key):
    if not root:
        return None
    if key < root.val:
        root.left = delete(root.left, key)
    elif key > root.val:
        root.right = delete(root.right, key)
    else:
        if not root.left:
            return root.right
        if not root.right:
            return root.left
        succ = root.right
        while succ.left:
            succ = succ.left
        root.val = succ.val
        root.right = delete(root.right, succ.val)
    return root

t = tree_from_list([5, 3, 6, 2, 4, None, 7])
t = delete(t, 3)
print(inorder(t))   # [2, 4, 5, 6, 7]

[2, 4, 5, 6, 7]


## 3. Validate BST

**Trap:** checking only `left.val < node.val < right.val` is wrong. Every node in the right subtree must be bigger than **all** ancestors it is on the right of. Pass an allowed **range** `(lo, hi)` down.

In [3]:
def is_valid_bst(root, lo=float("-inf"), hi=float("inf")):
    if not root:
        return True
    if not (lo < root.val < hi):
        return False
    return is_valid_bst(root.left, lo, root.val) and is_valid_bst(root.right, root.val, hi)

print(is_valid_bst(tree_from_list([2, 1, 3])))                    # True
print(is_valid_bst(tree_from_list([5, 1, 4, None, None, 3, 6])))  # False
print(is_valid_bst(tree_from_list([5, 4, 6, None, None, 3, 7])))  # False  (3 is in the right subtree of 5)

True
False
False


## 4. Kth Smallest in a BST

Inorder gives sorted order, so stop at the `k`-th visited node. Iterative inorder with early exit: O(h + k).

In [4]:
def kth_smallest(root, k):
    stack, cur = [], root
    while cur or stack:
        while cur:
            stack.append(cur)
            cur = cur.left
        cur = stack.pop()
        k -= 1
        if k == 0:
            return cur.val
        cur = cur.right

print(kth_smallest(tree_from_list([3, 1, 4, None, 2]), 1))                       # 1
print(kth_smallest(tree_from_list([5, 3, 6, 2, 4, None, None, 1]), 3))           # 3

1
3


## 5. LCA in a BST and Inorder Successor

- **LCA:** if both values are smaller → go left. Both bigger → go right. Otherwise the current node is the **split point** = LCA.
- **Inorder successor:** the smallest value **greater than** `p`. Walk down: if `p < node`, this node is a candidate (remember it) and go left; else go right.

In [5]:
def lca_bst(root, p, q):
    while root:
        if p < root.val and q < root.val:
            root = root.left
        elif p > root.val and q > root.val:
            root = root.right
        else:
            return root.val

def inorder_successor(root, p):
    succ = None
    while root:
        if p < root.val:
            succ = root
            root = root.left
        else:
            root = root.right
    return succ.val if succ else None

t = tree_from_list([6, 2, 8, 0, 4, 7, 9, None, None, 3, 5])
print(lca_bst(t, 2, 8), lca_bst(t, 2, 4))                       # 6 2
print(inorder_successor(tree_from_list([2, 1, 3]), 1), inorder_successor(tree_from_list([5, 3, 6, 2, 4, None, None, 1]), 6))   # 2 None

6 2
2 None


## 6. Range Sum of BST (pruning)

Sum of values in `[low, high]`. Use the BST rule to **skip** subtrees: if `node.val < low`, the left subtree is too small; if `node.val > high`, the right subtree is too big.

In [6]:
def range_sum_bst(root, low, high):
    if not root:
        return 0
    if root.val < low:
        return range_sum_bst(root.right, low, high)
    if root.val > high:
        return range_sum_bst(root.left, low, high)
    return root.val + range_sum_bst(root.left, low, high) + range_sum_bst(root.right, low, high)

print(range_sum_bst(tree_from_list([10, 5, 15, 3, 7, None, 18]), 7, 15))   # 32

32


## 7. BST Iterator (design)

Provide `next()` (next smallest) and `has_next()` in **O(1) amortized** time and **O(h) memory** without storing all values. Keep a stack with the **left chain**. On `next()`: pop, then push the left chain of its right child.

In [7]:
class BSTIterator:
    def __init__(self, root):
        self.stack = []
        self._push_left(root)

    def _push_left(self, node):
        while node:
            self.stack.append(node)
            node = node.left

    def next(self):
        node = self.stack.pop()
        self._push_left(node.right)
        return node.val

    def has_next(self):
        return bool(self.stack)

it = BSTIterator(tree_from_list([7, 3, 15, None, None, 9, 20]))
out = []
while it.has_next():
    out.append(it.next())
print(out)   # [3, 7, 9, 15, 20]

[3, 7, 9, 15, 20]


## 8. Interview thinking

**Signals:** "BST", "sorted", "k-th smallest", "range", "closest value", "successor", "validate".

**Say:** "The BST property lets me discard half the tree at each step (O(h)). Inorder gives sorted order."

**Common mistakes:** validating only parent-child instead of the full range, using `<=` vs `<` wrongly with duplicates, forgetting the two-children delete case, assuming O(log n) without balance, mutating the tree during iteration.

## Quick Revision

- BST: left < node < right at **every** node. Inorder = sorted.
- Search/insert/delete O(h); balanced O(log n), skewed O(n).
- Delete: leaf / one child / two children (inorder successor).
- Validate with a `(lo, hi)` range.
- LCA: split point. Successor: remember the last node where you went left.
- Iterator: stack of the left chain.